<a href="https://colab.research.google.com/github/Skaims/DLAV/blob/main/DLAV_Phase2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2: Trajectory Prediction with Auxiliary Depth Estimation

# 🧭 Introduction

"""
Welcome to **Phase 2** of the DLAV Projec! 🚗💨

In this phase, you'll work with a more challenging dataset that includes:
- RGB **camera images**
- Ground-truth **depth maps**
- Ground-truth **semantic segmentation** labels

Your goal is still to predict the **future trajectory** of the self-driving car (SDC), but you now have more tools at your disposal! 🎯

Here, we provide an example where **depth estimation** is used as an auxiliary task to improve trajectory prediction.

However, you're **free to explore** other auxiliary tasks (e.g., using semantic labels), different loss functions, data augmentations, or better architectures! 💡

This notebook will walk you through loading the dataset, building a model, training with and without the auxiliary task, and visualizing results.
"""

In [ ]:
# Install gdown to handle Google Drive file download
!pip install -q gdown

import gdown
import zipfile

download_url = f"https://drive.google.com/uc?id=1YkGwaxBKNiYL2nq--cB6WMmYGzRmRKVr"
output_zip = "dlav_train.zip"
gdown.download(download_url, output_zip, quiet=False)  # Downloads the file to your drive
with zipfile.ZipFile(output_zip, 'r') as zip_ref:  # Extracts the downloaded zip file
    zip_ref.extractall(".")

download_url = "https://drive.google.com/uc?id=1wtmT_vH9mMUNOwrNOMFP6WFw6e8rbOdu"
output_zip = "dlav_val.zip"
gdown.download(download_url, output_zip, quiet=False)
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall(".")

download_url = "https://drive.google.com/uc?id=1G9xGE7s-Ikvvc2-LZTUyuzhWAlNdLTLV"
output_zip = "dlav_test_public.zip"
gdown.download(download_url, output_zip, quiet=False)
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall(".")

Downloading...
From (original): https://drive.google.com/uc?id=1YkGwaxBKNiYL2nq--cB6WMmYGzRmRKVr
From (redirected): https://drive.google.com/uc?id=1YkGwaxBKNiYL2nq--cB6WMmYGzRmRKVr&confirm=t&uuid=5b75249d-8188-4173-81fa-58aa62a2b6db
To: /content/dlav_train.zip
100%|██████████| 439M/439M [00:03<00:00, 131MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1wtmT_vH9mMUNOwrNOMFP6WFw6e8rbOdu
From (redirected): https://drive.google.com/uc?id=1wtmT_vH9mMUNOwrNOMFP6WFw6e8rbOdu&confirm=t&uuid=0a980242-c4a4-420f-8c1a-a92788cf61b7
To: /content/dlav_val.zip
100%|██████████| 87.8M/87.8M [00:02<00:00, 30.5MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1G9xGE7s-Ikvvc2-LZTUyuzhWAlNdLTLV
From (redirected): https://drive.google.com/uc?id=1G9xGE7s-Ikvvc2-LZTUyuzhWAlNdLTLV&confirm=t&uuid=8c42f62b-0273-4296-8fd0-b8c695152ea9
To: /content/dlav_test_public.zip
100%|██████████| 86.6M/86.6M [00:00<00:00, 130MB/s]


## 📂 The Dataset

We are now working with a richer dataset that includes not just images and trajectories,
but also **depth maps** (and semantic segmentation labels, though unused in this example).

The data is stored in `.pkl` files and each file contains:
- `camera`: RGB image (shape: H x W x 3)
- `sdc_history_feature`: the past trajectory of the car
- `sdc_future_feature`: the future trajectory to predict
- `depth`: ground truth depth map (shape: H x W x 1)

We'll define a `DrivingDataset` class to load and return these tensors in a format our model can work with.

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
import pickle
from tqdm import tqdm

class DrivingDataset(Dataset):
    def __init__(self, file_list, test=False, augment=False):
        self.samples = file_list
        self.test    = test
        self.augment = augment   # seulement pour le train set

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        with open(self.samples[idx], 'rb') as f:
            data = pickle.load(f)

        camera  = torch.from_numpy(data['camera']).permute(2, 0, 1).float() / 255.0
        history = torch.tensor(data['sdc_history_feature'], dtype=torch.float32)
        future  = None if self.test else torch.tensor(data['sdc_future_feature'], dtype=torch.float32)

        depth = torch.from_numpy(data['depth']).float()
        if depth.ndim == 3 and depth.shape[-1] == 1:
            depth = depth.squeeze(-1)
        depth = depth.unsqueeze(0)

        seg = torch.from_numpy(data['semantic_label']).long()

        # Centrage autour du dernier point d'historique
        last_pos = history[-1, :2].clone()
        history[:, :2] -= last_pos
        if future is not None:
            future[:, :2] -= last_pos

        # ── Data augmentation : flip horizontal ───────────────────────
        # On retourne l'image ET on inverse le signe de x dans les trajectoires
        if self.augment and not self.test and random.random() < 0.5:
            camera = TF.hflip(camera)
            depth  = TF.hflip(depth)
            seg    = TF.hflip(seg.unsqueeze(0)).squeeze(0)
            history[:, 0] = -history[:, 0]
            if future is not None:
                future[:, 0] = -future[:, 0]

        out = {'camera': camera, 'history': history, 'depth': depth, 'seg': seg}
        if not self.test:
            out['future'] = future
        return out


## 🧠 The Model: Trajectory + Depth Prediction

We've extended our trajectory prediction model to optionally include a **depth estimation decoder**.

Why?
- Predicting depth helps the model **learn richer visual features** from the camera input.
- This acts as a form of **multi-task learning**, where learning to estimate depth reinforces scene understanding, ultimately leading to better trajectory predictions.
- This can be especially useful in complex environments with occlusions or sharp turns.

The model has:
- A CNN backbone to extract features from the image
- An MLP to process historical trajectory features
- A trajectory decoder to predict future coordinates
- (Optionally) A depth decoder to predict dense depth maps

This auxiliary task is enabled by setting `use_depth_aux=True`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DrivingPlanner(nn.Module):
    def __init__(self, use_depth_aux=False):
        super().__init__()
        self.use_depth_aux = use_depth_aux

        self.cnn_backbone = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2),
            nn.ReLU()
        )

        self.cnn_flatten = nn.Flatten()


        # Decoder for trajectory prediction
        self.decoder = nn.Linear((32 * 100 * 150) + (21 * 3), 60 * 3)

        # Optional depth decoder
        if self.use_depth_aux:
            self.depth_decoder = nn.Sequential(
                nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),  # Upsample
                nn.ReLU(),
                nn.Conv2d(16, 1, kernel_size=3, padding=1),
                nn.Upsample(size=(200, 300), mode='bilinear', align_corners=False)
            )

    def forward(self, camera, history):
        B = camera.size(0)

        # Process camera
        cnn_feat = self.cnn_backbone(camera)         # (B, 32, 100, 150)
        feat_flat = self.cnn_flatten(cnn_feat)       # (B, 32*100*150)


        # Flatten history
        history_flat = history.view(B, -1)

        # Concatenate and decode trajectory
        combined = torch.cat([feat_flat, history_flat], dim=1)
        future = self.decoder(combined).view(B, 60, 3)

        # Optional depth map prediction
        depth_out = None
        if self.use_depth_aux:
            depth_out = self.depth_decoder(cnn_feat.detach()).permute(0, 2, 3, 1)

        return future, depth_out


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


# ─────────────────────────────────────────────────────────────────────────────
# FPN léger (Feature Pyramid Network)
# ─────────────────────────────────────────────────────────────────────────────

class LightFPN(nn.Module):
    """
    Fusionne layer2 (128ch), layer3 (256ch), layer4 (512ch) en une
    carte unique de dimension feat_dim à la résolution de layer3 (/16).
    """
    def __init__(self, feat_dim=256):
        super().__init__()
        self.lat4 = nn.Conv2d(512, feat_dim, 1)
        self.lat3 = nn.Conv2d(256, feat_dim, 1)
        self.lat2 = nn.Conv2d(128, feat_dim, 1)
        self.out  = nn.Sequential(
            nn.Conv2d(feat_dim, feat_dim, 3, padding=1),
            nn.BatchNorm2d(feat_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, s2, s3, s4):
        """Retourne feature map à la résolution de s3."""
        p4 = self.lat4(s4)
        p4_up = F.interpolate(p4, size=s3.shape[2:], mode='nearest')
        p3 = self.lat3(s3) + p4_up
        p3_up = F.interpolate(p3, size=s2.shape[2:], mode='nearest')
        p2 = self.lat2(s2) + p3_up
        # On retourne à la résolution de s3 (/16)
        p2_down = F.adaptive_avg_pool2d(p2, s3.shape[2:])
        fused = p3 + p2_down
        return self.out(fused)   # (B, feat_dim, H/16, W/16)


# ─────────────────────────────────────────────────────────────────────────────
# Encodeur de depth (maps → vecteur de features)
# ─────────────────────────────────────────────────────────────────────────────

class DepthEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 5, stride=2, padding=2), nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(64 * 16, out_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, depth):
        """depth : (B, 1, H, W) in meters"""
        # Normalisation log pour compresser la dynamique
        return self.net(torch.log1p(depth))


# ─────────────────────────────────────────────────────────────────────────────
# U-Net UpBlock pour les tâches auxiliaires
# ─────────────────────────────────────────────────────────────────────────────

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch // 2 + skip_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


# ─────────────────────────────────────────────────────────────────────────────
# Waypoint Query Decoder  (le cœur du changement)
# ─────────────────────────────────────────────────────────────────────────────

class WaypointQueryDecoder(nn.Module):
    """
    T queries learnable (une par timestep futur) font de la cross-attention
    sur la feature map visuelle + contexte global via self-attention.

    Architecture : N couches de (Self-Attn → Cross-Attn → FFN)
    Prédiction finale : coordonnées ABSOLUES (relatives au dernier point hist)
    """
    def __init__(self, future_steps=60, future_dim=3,
                 d_model=256, nhead=8, num_layers=3,
                 feat_h=12, feat_w=20, dropout=0.1):
        super().__init__()
        self.future_steps = future_steps
        self.future_dim   = future_dim
        self.d_model      = d_model

        # T queries learnable : chaque query ≈ "que se passe-t-il au temps t ?"
        self.waypoint_queries = nn.Embedding(future_steps, d_model)

        # Encodage positionnel sinusoïdal pour la feature map 2D
        self.register_buffer(
            'pos_enc',
            self._make_pos_enc(feat_h, feat_w, d_model)
        )

        # Couches Transformer decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
            norm_first=True,   # Pre-norm : plus stable
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        # Tête de sortie : prédit (x, y, [z]) absolus
        self.out_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(inplace=True),
            nn.Linear(d_model // 2, future_dim),
        )

    @staticmethod
    def _make_pos_enc(H, W, d_model):
        """Encodage positionnel 2D pour la feature map (H*W, d_model)."""
        y_enc = torch.zeros(H, d_model // 2)
        x_enc = torch.zeros(W, d_model // 2)
        div   = torch.exp(torch.arange(0, d_model // 2, 2).float() * -(torch.log(torch.tensor(10000.0)) / (d_model // 2)))
        y_pos = torch.arange(H).float().unsqueeze(1)
        x_pos = torch.arange(W).float().unsqueeze(1)
        y_enc[:, 0::2] = torch.sin(y_pos * div)
        y_enc[:, 1::2] = torch.cos(y_pos * div)
        x_enc[:, 0::2] = torch.sin(x_pos * div)
        x_enc[:, 1::2] = torch.cos(x_pos * div)
        # Combine : (H, W, d_model)
        pos = torch.cat([
            y_enc.unsqueeze(1).expand(H, W, d_model // 2),
            x_enc.unsqueeze(0).expand(H, W, d_model // 2),
        ], dim=-1)
        return pos.view(H * W, d_model)   # (HW, d_model)

    def forward(self, feat_map, context):
        """
        feat_map : (B, d_model, H', W')  — feature map FPN
        context  : (B, d_model)          — vecteur global (img+hist+depth)

        Retourne : (B, T, future_dim)  coordonnées absolues
        """
        B = feat_map.size(0)
        _, C, H, W = feat_map.shape

        # Flatten la feature map → séquence de mémoire pour le cross-attention
        mem = feat_map.flatten(2).permute(0, 2, 1)   # (B, HW, d_model)

        # Ajout encodage positionnel (adapté dynamiquement à la taille réelle)
        if H * W != self.pos_enc.shape[0]:
            pos = F.interpolate(
                self.pos_enc.view(1, *self.pos_enc.shape).permute(0, 2, 1).unsqueeze(-1),
                size=(H * W, 1), mode='bilinear', align_corners=False
            ).squeeze().permute(1, 0)   # (HW, d_model)
        else:
            pos = self.pos_enc
        mem = mem + pos.unsqueeze(0)

        # Queries : T waypoints, conditionnés par le contexte global
        queries = self.waypoint_queries.weight.unsqueeze(0).expand(B, -1, -1)  # (B, T, d_model)
        queries = queries + context.unsqueeze(1)   # Injection du contexte global

        # Transformer decoder
        out = self.transformer_decoder(queries, mem)   # (B, T, d_model)

        return self.out_head(out)   # (B, T, future_dim)  — coordonnées absolues


# ─────────────────────────────────────────────────────────────────────────────
# Modèle principal
# ─────────────────────────────────────────────────────────────────────────────

class PretrainedResNetPlannerCMD(nn.Module):
    def __init__(self,
                 history_steps: int = 21,
                 future_steps: int = 60,
                 history_dim: int = 3,
                 future_dim: int = 3,
                 backbone_pretrained: bool = True,
                 history_hidden: int = 128,
                 d_model: int = 256,
                 nhead: int = 8,
                 num_decoder_layers: int = 3,
                 dropout: float = 0.1,
                 num_seg_classes: int = 15,
                 use_depth_aux: bool = True,
                 use_seg_aux: bool = True,
                 feat_h: int = 12,    # H/16 pour image 192px
                 feat_w: int = 20):   # W/16 pour image 320px
        super().__init__()

        self.future_steps  = future_steps
        self.future_dim    = future_dim
        self.d_model       = d_model
        self.use_depth_aux = use_depth_aux
        self.use_seg_aux   = use_seg_aux

        # ── Backbone ResNet-34 ───────────────────────────────────────────
        resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if backbone_pretrained else None)
        self.stem    = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.maxpool = resnet.maxpool
        self.layer1  = resnet.layer1   #  64ch /4
        self.layer2  = resnet.layer2   # 128ch /8
        self.layer3  = resnet.layer3   # 256ch /16
        self.layer4  = resnet.layer4   # 512ch /32

        # ── FPN → d_model channels ──────────────────────────────────────
        self.fpn = LightFPN(feat_dim=d_model)

        # ── Encodeur image global ────────────────────────────────────────
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.img_proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(inplace=True),
        )

        # ── Encodeur depth (utilisé comme feature, pas seulement loss) ──
        self.depth_encoder = DepthEncoder(out_dim=d_model // 2)
        self.depth_feat_proj = nn.Linear(d_model // 2, d_model)

        # ── GRU historique ───────────────────────────────────────────────
        self.history_gru = nn.GRU(
            input_size=history_dim, hidden_size=history_hidden,
            num_layers=2, batch_first=True,
            dropout=dropout, bidirectional=True,
        )
        self.hist_proj = nn.Sequential(
            nn.Linear(history_hidden * 2, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(inplace=True),
        )

        # ── Fusion : img + hist + depth → context ───────────────────────
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 3, d_model * 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model),
        )

        # ── Waypoint Query Decoder ───────────────────────────────────────
        self.waypoint_decoder = WaypointQueryDecoder(
            future_steps=future_steps,
            future_dim=future_dim,
            d_model=d_model,
            nhead=nhead,
            num_layers=num_decoder_layers,
            feat_h=feat_h,
            feat_w=feat_w,
            dropout=dropout,
        )

        # ── Tâches auxiliaires (U-Net) ───────────────────────────────────
        self.bottleneck_proj = nn.Conv2d(512, 256, 1)
        self.up1 = UpBlock(256, 256, 128)
        self.up2 = UpBlock(128, 128, 64)
        self.up3 = UpBlock(64,  64,  32)

        if self.use_depth_aux:
            self.depth_head = nn.Sequential(
                nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(inplace=True),
                nn.Conv2d(32, 1, 1), nn.Softplus(),
            )
        if self.use_seg_aux:
            self.seg_head = nn.Sequential(
                nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True),
                nn.Conv2d(64, num_seg_classes, 1),
            )

        # ── Normalisation ImageNet ───────────────────────────────────────
        self.register_buffer("img_mean", torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1), persistent=False)
        self.register_buffer("img_std",  torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1), persistent=False)

    # ── Encodeurs ────────────────────────────────────────────────────────────

    def encode_image(self, camera):
        x  = (camera - self.img_mean) / self.img_std
        s0 = self.stem(x)
        x  = self.maxpool(s0)
        s1 = self.layer1(x)
        s2 = self.layer2(s1)
        s3 = self.layer3(s2)
        e4 = self.layer4(s3)
        img_global = self.img_proj(self.global_pool(e4).flatten(1))
        feat_map   = self.fpn(s2, s3, e4)   # (B, d_model, H/16, W/16)
        skips = dict(s1=s1, s2=s2, s3=s3, e4=e4)
        return skips, img_global, feat_map

    def encode_history(self, history):
        _, h_n = self.history_gru(history)
        return self.hist_proj(torch.cat([h_n[-2], h_n[-1]], dim=1))

    def decode_spatial(self, skips):
        e4 = self.bottleneck_proj(skips['e4'])
        d  = self.up1(e4,  skips['s3'])
        d  = self.up2(d,   skips['s2'])
        d  = self.up3(d,   skips['s1'])
        return d

    # ── Forward ───────────────────────────────────────────────────────────────

    def forward(self, camera, history, depth=None,
                teacher_forcing_prob=0.0, gt_future=None):
        """
        camera  : (B, 3, H, W)  in [0, 1]
        history : (B, T_h, 3)
        depth   : (B, 1, H, W)  in meters  ← utilisé comme feature
        """
        skips, img_feat, feat_map = self.encode_image(camera)
        hist_feat = self.encode_history(history)

        # Depth comme feature d'entrée (si disponible)
        if depth is not None:
            depth_feat = self.depth_feat_proj(self.depth_encoder(depth))
        else:
            depth_feat = torch.zeros_like(img_feat)

        context = self.fusion(torch.cat([img_feat, hist_feat, depth_feat], dim=1))

        # Prédiction de trajectoire : coordonnées absolues directes
        traj = self.waypoint_decoder(feat_map, context)   # (B, T, future_dim)

        # Tâches auxiliaires
        depth_up = seg_up = None
        if self.use_depth_aux or self.use_seg_aux:
            d = self.decode_spatial(skips)
            d = F.interpolate(d, size=camera.shape[2:], mode='bilinear', align_corners=False)
            if self.use_depth_aux:
                depth_up = self.depth_head(d)
            if self.use_seg_aux:
                seg_up = self.seg_head(d)

        return traj, depth_up, seg_up

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Loss temporellement pondérée (inchangée)
# ─────────────────────────────────────────────────────────────────────────────

def temporal_traj_loss(pred, gt, T, device):
    huber   = nn.SmoothL1Loss(reduction='none', beta=0.5)
    weights = torch.linspace(0.5, 1.5, T, device=device)
    per_t   = huber(pred[..., :2], gt[..., :2]).sum(-1)   # (B, T)
    weighted = (per_t * weights).mean()
    way_idx  = torch.tensor([9, 19, 29, 44, 59], device=device)
    wp_loss  = huber(pred[:, way_idx, :2], gt[:, way_idx, :2]).mean()
    return weighted + 0.5 * wp_loss


# ─────────────────────────────────────────────────────────────────────────────
# Boucle d'entraînement
# ─────────────────────────────────────────────────────────────────────────────

def compute_ade_fde(pred, gt):
    err = torch.norm(pred[:, :, :2] - gt[:, :, :2], dim=2)
    return err.mean(dim=1).sum().item(), err[:, -1].sum().item(), pred.size(0)

    return weighted + 0.5 * wp_loss

In [ ]:
def train(model, train_loader, val_loader,
          num_epochs=60, lr=2e-4, device=None):

    device = device or (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
    model  = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr,
        steps_per_epoch=len(train_loader),
        epochs=num_epochs,
        pct_start=0.1, anneal_strategy='cos',
    )
    scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))
    ce_fn  = nn.CrossEntropyLoss()
    T      = model.future_steps

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Train {epoch+1}/{num_epochs}"):
            cam  = batch['camera'].to(device)
            hist = batch['history'].to(device)
            fut  = batch['future'].to(device)
            dep  = batch['depth'].to(device)
            seg  = batch.get('seg')
            if seg is not None:
                seg = seg.to(device)

            optimizer.zero_grad()
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                # Passage de la depth comme feature d'entrée
                fut_pred, dep_pred, seg_pred = model(cam, hist, depth=dep)

                loss = temporal_traj_loss(fut_pred, fut, T, device)

                if dep_pred is not None:
                    if dep_pred.shape[2:] != dep.shape[2:]:
                        dep_pred = F.interpolate(dep_pred, size=dep.shape[2:], mode='bilinear', align_corners=False)
                    loss = loss + 0.1 * F.l1_loss(dep_pred, dep)

                if seg_pred is not None and seg is not None:
                    loss = loss + 0.1 * ce_fn(seg_pred, seg)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item()

        # ── Validation ───────────────────────────────────────────────────
        model.eval()
        total_ade = total_fde = total_mse = 0.0
        total_seg_correct = total_seg_pixels = 0
        n = 0

        with torch.no_grad():
            for batch in val_loader:
                cam  = batch['camera'].to(device)
                hist = batch['history'].to(device)
                fut  = batch['future'].to(device)
                dep  = batch['depth'].to(device)
                seg  = batch.get('seg')
                if seg is not None:
                    seg = seg.to(device)

                fut_pred, _, seg_pred = model(cam, hist, depth=dep)
                ade, fde, cnt = compute_ade_fde(fut_pred, fut)
                total_ade += ade; total_fde += fde
                total_mse += F.mse_loss(fut_pred, fut, reduction='sum').item()
                n += cnt

                if seg_pred is not None and seg is not None:
                    preds = seg_pred.argmax(dim=1)
                    total_seg_correct += (preds == seg).sum().item()
                    total_seg_pixels  += preds.numel()

        print(
            f"Epoch {epoch+1}/{num_epochs} | "
            f"TrainLoss {running_loss/len(train_loader):.4f} | "
            f"Val ADE {total_ade/n:.4f} FDE {total_fde/n:.4f} "
            f"MSE {total_mse/(n*T*3):.4f} "
            f"SEG {total_seg_correct/max(1,total_seg_pixels):.4f} | "
            f"LR {optimizer.param_groups[0]['lr']:.2e}"
        )

In [41]:
import os
import torch.optim as optim

seed = 42
torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

NUM_SEG_CLASSES = 15
BATCH_SIZE      = 32
NUM_EPOCHS      = 60
LR              = 3e-4

train_data_dir = "train"
val_data_dir   = "val"
train_files = [os.path.join(train_data_dir, f) for f in os.listdir(train_data_dir) if f.endswith('.pkl')]
val_files   = [os.path.join(val_data_dir,   f) for f in os.listdir(val_data_dir)   if f.endswith('.pkl')]

train_dataset = DrivingDataset(train_files, augment=True)
val_dataset   = DrivingDataset(val_files,   augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# Auto-détection de la résolution
sample = next(iter(train_loader))
_, _, H, W = sample['camera'].shape
feat_h, feat_w = H // 16, W // 16
print(f"feat map : {feat_h}×{feat_w}")

model = PretrainedResNetPlannerCMD(
    history_steps=21,
    future_steps=60,
    d_model=256,
    nhead=8,
    num_decoder_layers=3,
    dropout=0.1,
    num_seg_classes=15,
    use_depth_aux=True,
    use_seg_aux=True,
    feat_h=feat_h,
    feat_w=feat_w,
)

train(model, train_loader, val_loader, num_epochs=60, lr=2e-4)


feat map : 12×18


Train 1/60: 100%|██████████| 157/157 [00:41<00:00,  3.75it/s]


Epoch 1/60 | TrainLoss 42.1732 | Val ADE 9.8699 FDE 19.8664 MSE 63.9441 SEG 0.8242 | LR 2.09e-05


Train 2/60: 100%|██████████| 157/157 [00:39<00:00,  3.98it/s]


Epoch 2/60 | TrainLoss 35.0440 | Val ADE 7.0994 FDE 10.6563 MSE 40.1275 SEG 0.8794 | LR 5.61e-05


...

Epoch 58/60 | TrainLoss 0.7129 | Val ADE 1.6698 FDE 4.4992 MSE 5.2653 SEG 0.9571 | LR 6.73e-07


Train 59/60: 100%|██████████| 157/157 [00:39<00:00,  3.96it/s]


Epoch 59/60 | TrainLoss 0.6970 | Val ADE 1.6651 FDE 4.4886 MSE 5.2330 SEG 0.9568 | LR 1.68e-07


Train 60/60: 100%|██████████| 157/157 [00:40<00:00,  3.89it/s]


Epoch 60/60 | TrainLoss 0.7084 | Val ADE 1.6649 FDE 4.4759 MSE 5.2528 SEG 0.9571 | LR 8.07e-10


## 🏋️ Training with Auxiliary Loss

The training loop is similar to Phase 1 — except now, if enabled, we also compute a loss on the predicted **depth map**.

We define:
- `trajectory_loss` as standard MSE between predicted and true future trajectory
- `depth_loss` as L1 loss between predicted and ground truth depth

Total loss = `trajectory_loss + lambda * depth_loss`

This helps guide the model to learn better representations from visual input. The weight `lambda` is a hyperparameter you can tune!

In [ ]:
import torch
import torch.nn.functional as F

def train_one_epoch(model, train_loader, optimizer, device, lambda_depth=0.1, use_depth_aux=False):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        cam, hist, fut, dep = [batch[k].to(device) for k in ['camera', 'history', 'future', 'depth']]
        optimizer.zero_grad()
        fut_pred, dep_pred = model(cam, hist)

        traj_loss = F.mse_loss(fut_pred, fut)
        loss = traj_loss
        if use_depth_aux:
            loss += lambda_depth * F.l1_loss(dep_pred, dep)

        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_loss = train_loss / len(train_loader)
    return avg_loss

def validate(model, val_loader, device):
    model.eval()
    total_ade, total_fde, total_mse = 0.0, 0.0, 0.0
    count = 0

    with torch.no_grad():
        for batch in val_loader:
            cam = batch['camera'].to(device)
            hist = batch['history'].to(device)
            fut = batch['future'].to(device)

            fut_pred, _ = model(cam, hist)

            B, T, _ = fut.shape
            count += B

            ade = torch.norm(fut_pred[:, :, :2] - fut[:, :, :2], dim=2).mean(dim=1).sum()
            fde = torch.norm(fut_pred[:, -1, :2] - fut[:, -1, :2], dim=1).sum()
            mse = F.mse_loss(fut_pred, fut, reduction='sum')

            total_ade += ade.item()
            total_fde += fde.item()
            total_mse += mse.item()

    ade_avg = total_ade / count
    fde_avg = total_fde / count
    mse_avg = total_mse / (count * T * 3)

    return ade_avg, fde_avg, mse_avg

def train(model, train_loader, val_loader, optimizer, num_epochs=50, lambda_depth=0.1, use_depth_aux=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    model = model.to(device)

    for epoch in range(num_epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, device, lambda_depth, use_depth_aux)
        ade, fde, mse = validate(model, val_loader, device)

        print(f"Epoch {epoch+1}, Loss: {train_loss:.4f}, Validation - ADE: {ade:.4f}, FDE: {fde:.4f}, Traj MSE: {mse:.6f}")


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
import os

train_data_dir = "train"
val_data_dir = "val"

train_files = [os.path.join(train_data_dir, f) for f in os.listdir(train_data_dir) if f.endswith('.pkl')]
val_files = [os.path.join(val_data_dir, f) for f in os.listdir(val_data_dir) if f.endswith('.pkl')]

train_dataset = DrivingDataset(train_files)
val_dataset = DrivingDataset(val_files)

train_loader = DataLoader(train_dataset, batch_size=32, num_workers=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=2)


## 🔍 Let's Compare Two Settings

We'll now train and evaluate the model in **two modes**:

1. **Without auxiliary task** — the model only predicts the trajectory.
2. **With depth auxiliary task** — the model also predicts a depth map, which helps it learn better visual features.

By comparing the results (ADE, FDE, and Trajectory MSE), you'll see the benefit of multi-task learning in action! 🚀

In [ ]:
use_depth_aux=False
model_no_aux = DrivingPlanner(use_depth_aux=use_depth_aux)
optimizer = optim.Adam(model_no_aux.parameters(), lr=1e-3)
train(model_no_aux, train_loader, val_loader, optimizer, num_epochs=50,use_depth_aux=use_depth_aux)

Using device: cuda
Epoch 1, Loss: 87.4480, Validation - ADE: 5.2175, FDE: 13.0018, Traj MSE: 18.809232
Epoch 2, Loss: 14.3163, Validation - ADE: 3.9738, FDE: 10.0709, Traj MSE: 12.100130
Epoch 3, Loss: 10.4606, Validation - ADE: 3.6965, FDE: 9.1975, Traj MSE: 10.717365
Epoch 4, Loss: 8.6864, Validation - ADE: 3.8241, FDE: 9.0263, Traj MSE: 10.878973
Epoch 5, Loss: 7.3263, Validation - ADE: 3.7802, FDE: 9.0681, Traj MSE: 10.754304
Epoch 6, Loss: 6.1287, Validation - ADE: 3.6971, FDE: 8.8897, Traj MSE: 10.401305
Epoch 7, Loss: 5.2579, Validation - ADE: 3.9032, FDE: 8.9781, Traj MSE: 11.092262
Epoch 8, Loss: 4.4888, Validation - ADE: 3.8602, FDE: 8.9591, Traj MSE: 10.981510
Epoch 9, Loss: 3.7284, Validation - ADE: 3.8807, FDE: 9.0030, Traj MSE: 11.118335
Epoch 10, Loss: 3.3431, Validation - ADE: 3.8664, FDE: 9.0330, Traj MSE: 11.094842
Epoch 11, Loss: 2.8673, Validation - ADE: 3.8949, FDE: 9.0028, Traj MSE: 11.157735
Epoch 12, Loss: 2.4861, Validation - ADE: 3.8835, FDE: 9.0434, Traj MSE:

In [ ]:
use_depth_aux=True
model_with_aux = DrivingPlanner(use_depth_aux=use_depth_aux)
optimizer = optim.Adam(model_with_aux.parameters(), lr=1e-3)
train(model_with_aux, train_loader, val_loader, optimizer, num_epochs=50,use_depth_aux=use_depth_aux, lambda_depth=0.05)

Using device: cuda
Epoch 1, Loss: 101.1327, Validation - ADE: 5.1595, FDE: 12.6716, Traj MSE: 18.597009
Epoch 2, Loss: 25.0148, Validation - ADE: 4.0220, FDE: 10.3022, Traj MSE: 12.502671
Epoch 3, Loss: 18.2951, Validation - ADE: 3.6960, FDE: 9.2855, Traj MSE: 10.818774
Epoch 4, Loss: 12.4377, Validation - ADE: 3.9746, FDE: 9.3555, Traj MSE: 11.444528
Epoch 5, Loss: 8.5513, Validation - ADE: 3.7805, FDE: 8.9672, Traj MSE: 10.715357
Epoch 6, Loss: 6.8699, Validation - ADE: 3.9320, FDE: 9.1462, Traj MSE: 11.231993
Epoch 7, Loss: 5.9278, Validation - ADE: 3.8306, FDE: 8.9915, Traj MSE: 10.733713
Epoch 8, Loss: 5.0201, Validation - ADE: 3.9241, FDE: 9.2057, Traj MSE: 11.245680
Epoch 9, Loss: 4.5932, Validation - ADE: 3.8350, FDE: 8.9870, Traj MSE: 10.883760
Epoch 10, Loss: 3.9220, Validation - ADE: 3.9718, FDE: 9.1147, Traj MSE: 11.308907
Epoch 11, Loss: 3.5340, Validation - ADE: 3.8987, FDE: 9.1131, Traj MSE: 11.133181
Epoch 12, Loss: 3.2612, Validation - ADE: 3.8787, FDE: 9.0431, Traj MS

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ade, fde, mse = validate(model_no_aux, val_loader, device)
print(f"Validation results for model without depth auxiliary loss: ADE: {ade:.4f}, FDE: {fde:.4f}, Traj MSE: {mse:.6f}")

ade, fde, mse = validate(model_with_aux, val_loader, device)
print(f"Validation results for model with depth auxiliary loss: ADE: {ade:.4f}, FDE: {fde:.4f}, Traj MSE: {mse:.6f}")

Validation results for model without depth auxiliary loss: ADE: 3.8855, FDE: 9.4077, Traj MSE: 11.732960
Validation results for model with depth auxiliary loss: ADE: 3.7726, FDE: 9.1269, Traj MSE: 11.130413


## 🔍 Final Visualization and Comparison

Now that we’ve trained two models — one **with** the depth auxiliary task and one **without** — let’s visualize and compare their predictions.

We’ll show:
1. The **camera image** from selected validation examples
2. The **past trajectory**, **ground-truth future**, and **predicted future** trajectory
3. The **predicted vs. ground-truth depth maps** (only for the model trained with the auxiliary task)

These visualizations help us understand:
- Does the predicted trajectory better match the future when the depth task is included?
- Is the predicted depth map reasonably accurate?

Let’s see the difference! 📈

In [42]:
import matplotlib.pyplot as plt
import random

random.seed(40)

def visualize_comparison(val_loader, model_no_aux, model_with_aux, device):
    model_no_aux.eval()
    model_with_aux.eval()
    val_batch = next(iter(val_loader))

    camera = val_batch['camera'].to(device)
    history = val_batch['history'].to(device)
    future = val_batch['future'].to(device)
    depth = val_batch['depth'].to(device)

    with torch.no_grad():
        pred, _ = model(camera, history)

    camera = camera.cpu().numpy()
    history = history.cpu().numpy()
    future = future.cpu().numpy()
    pred= pred.cpu().numpy()
    depth = depth.cpu().numpy()
    pred_depth = pred_depth.cpu().numpy() if pred_depth is not None else None

    k = 4
    indices = random.choices(np.arange(len(camera)), k=k)

    # Show the input camera images
    fig, ax = plt.subplots(1, k, figsize=(4 * k, 4))
    for i, idx in enumerate(indices):
        ax[i].imshow(camera[idx].transpose(1, 2, 0))
        ax[i].set_title(f"Example {i+1}")
        ax[i].axis("off")
    plt.suptitle("Camera Inputs")
    plt.tight_layout()
    plt.show()

    # Compare predicted trajectories
    fig, ax = plt.subplots(2, k, figsize=(4 * k, 8))
    for i, idx in enumerate(indices):
        # Without aux
        ax[0, i].plot(history[idx, :, 0], history[idx, :, 1], 'o-', label='Past', color='gold', markersize=4, linewidth=1.2)
        ax[0, i].plot(future[idx, :, 0], future[idx, :, 1], 'o-', label='GT Future', color='green', markersize=4, linewidth=1.2)
        ax[0, i].plot(pred_no_aux[idx, :, 0], pred_no_aux[idx, :, 1], 'o-', label='Pred (No Aux)', color='red', markersize=4, linewidth=1.2)
        ax[0, i].set_title("No Depth Aux")
        ax[0, i].axis("equal")

        # With aux
        ax[1, i].plot(history[idx, :, 0], history[idx, :, 1], 'o-', label='Past', color='gold', markersize=4, linewidth=1.2)
        ax[1, i].plot(future[idx, :, 0], future[idx, :, 1], 'o-', label='GT Future', color='green', markersize=4, linewidth=1.2)
        ax[1, i].plot(pred_with_aux[idx, :, 0], pred_with_aux[idx, :, 1], 'o-', label='Pred (With Aux)', color='blue', markersize=4, linewidth=1.2)
        ax[1, i].set_title("With Depth Aux")
        ax[1, i].axis("equal")

    # Show full legend in a new figure
    fig_legend = plt.figure(figsize=(8, 1))
    legend_handles = [
        plt.Line2D([0], [0], color='gold', marker='o', linestyle='-', markersize=5, label='Past'),
        plt.Line2D([0], [0], color='green', marker='o', linestyle='-', markersize=5, label='GT Future'),
        plt.Line2D([0], [0], color='red', marker='o', linestyle='-', markersize=5, label='Pred (No Aux)'),
        plt.Line2D([0], [0], color='blue', marker='o', linestyle='-', markersize=5, label='Pred (With Aux)')
    ]
    fig_legend.legend(handles=legend_handles, loc='center', ncol=4)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    plt.suptitle("Trajectory Prediction: Without vs With Depth Aux Task")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    # Show predicted vs GT depth (only for bottom row)
    if pred_depth is not None:
        fig, ax = plt.subplots(2, k, figsize=(4 * k, 6))
        for i, idx in enumerate(indices):
            ax[0, i].imshow(depth[idx, :, :, 0], cmap='viridis')
            ax[0, i].set_title("GT Depth", pad=10)
            ax[0, i].axis("off")
            # increase vertical distance between rows

            ax[1, i].imshow(pred_depth[idx, :, :, 0], cmap='viridis')
            ax[1, i].set_title("Pred Depth", pad=10)
            ax[1, i].axis("off")

        plt.suptitle("Depth Estimation (Only for Model With Aux Task)", y=1.05)
        plt.subplots_adjust(hspace=0.4)
        plt.tight_layout()
        plt.show()


# 🔚 Call at the end after training both models
visualize_comparison(val_loader, model_no_aux, model_with_aux, device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

NameError: name 'model_no_aux' is not defined

Now we run our model on the test set once, to get the plan of our model and save it for submission. Notice that the ground truth plans are removed for the test set, so you can not calculate the ADE metric on the test set yourself, and need to submit it to the leader board. By running the last cell, you'll be able to see a csv file called submission_phase2.csv by clicking on the folder icon on the left. Download it and submit it to the leaderboard to get your score.

In [43]:
with open(f"test_public/0.pkl", "rb") as f:
    data = pickle.load(f)
print(data.keys())
# Note the absence of sdc_future_feature

dict_keys(['camera', 'depth', 'driving_command', 'sdc_history_feature', 'semantic_label'])


In [47]:
import pandas as pd
test_data_dir = "test_public"
test_files = [os.path.join(test_data_dir, fn) for fn in sorted([f for f in os.listdir(test_data_dir) if f.endswith(".pkl")], key=lambda fn: int(os.path.splitext(fn)[0]))]
test_dataset = DrivingDataset(test_files, test=True)
test_loader = DataLoader(test_dataset, batch_size=250, num_workers=2)
device = (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
model.eval()
all_plans = []
with torch.no_grad():
    for batch in test_loader:
        camera = batch['camera'].to(device)
        history = batch['history'].to(device)
        dep = batch['depth'].to(device)
        seg = batch.get('seg')
        if seg is not None:
            seg = seg.to(device)

        pred_future, _, _ = model(camera, history, depth=dep)
        all_plans.append(pred_future.cpu().numpy()[..., :2])
all_plans = np.concatenate(all_plans, axis=0)

# Now save the plans as a csv file
pred_xy = all_plans[..., :2]  # shape: (total_samples, T, 2)

# Flatten to (total_samples, T*2)
total_samples, T, D = pred_xy.shape
pred_xy_flat = pred_xy.reshape(total_samples, T * D)

# Build a DataFrame with an ID column
ids = np.arange(total_samples)
df_xy = pd.DataFrame(pred_xy_flat)
df_xy.insert(0, "id", ids)

# Column names: id, x_1, y_1, x_2, y_2, ..., x_T, y_T
new_col_names = ["id"]
for t in range(1, T + 1):
    new_col_names.append(f"x_{t}")
    new_col_names.append(f"y_{t}")
df_xy.columns = new_col_names

# Save to CSV
df_xy.to_csv("submission_phase2.csv", index=False)

print(f"Shape of df_xy: {df_xy.shape}")

Shape of df_xy: (1000, 121)
